In [2]:
from unittest.mock import AsyncMock, patch

import app.workflow_best2 as wf

locked_path = str(wf.DOCS_ROOT_DIR / "sops" / "direct_001.docx")
state = wf.WorkflowState(
    conversation_id="notebook-lock-test",
    original_user_query="Alder Supply",
    current_index_path=wf.ROOT_INDEX_PATH,
    selected_link=locked_path,
    active_sop_path=locked_path,
    locked_company_name="Alder Supply",
    locked_on_sop=True,
    status="locked_on_sop",
)

with (
    patch.object(
        wf.retrieval_agent,
        "run",
        new_callable=AsyncMock,
        side_effect=AssertionError("BUG: company router ran after the SOP was locked"),
    ) as router,
    patch.object(
        wf,
        "answer_from_sop",
        new_callable=AsyncMock,
        return_value=wf.SopAnswer(answer="Notebook test: answered from locked SOP."),
    ) as answer,
):
    # Natural-language lookup must stay with the existing SOP.
    message = "pull me wintergreen logistics"
    result = await wf.handle_turn(message, state)
    answer.assert_awaited_once_with(message, locked_path)
    assert result.state.active_sop_path == locked_path

    # Explicit switch must be rejected before routing or SOP retrieval.
    answer.reset_mock()
    result = await wf.handle_turn(
        "switch company: Wintergreen Logistics", result.state
    )
    router.assert_not_awaited()
    answer.assert_not_awaited()
    assert result.state.locked_on_sop
    assert result.state.active_sop_path == locked_path
    assert result.state.locked_company_name == "Alder Supply"
    print(result.assistant_text)
    print("PASS: the company and SOP stayed locked.")

This chat is locked to the selected company's SOP. I can't switch companies within this chat.
PASS: the company and SOP stayed locked.


In [1]:
import app.workflow_best2 as wf

print(wf.__file__)
print(any(
    isinstance(value, str) and "I can't switch companies" in value
    for value in wf.handle_turn.__code__.co_consts
))

/home/aytu/apps/docsearch/app/workflow_best2.py
True


In [1]:
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

import app.workflow_best2 as wf

wf.DOCS_ROOT_DIR = Path("data").resolve()
wf.ROOT_INDEX_PATH = str(wf.DOCS_ROOT_DIR / "index.xlsx")

# An address in the original question should resolve in one turn.
one_turn = await wf.handle_turn(
    "Is Cross OpCo allowed for Marigold Kitchens "
    "at 1010 Harbor Avenue, East Harbor?",
    state=None,
)

# Without an address, ask once and use the returned state.
first = await wf.handle_turn(
    "Is Cross OpCo allowed for Marigold Kitchens?",
    state=None,
)
assert first.event_type == "ask_user", first.assistant_text
assert first.state.in_secondary_index
assert first.state.secondary_company_name == "Marigold Kitchens"

two_turn = await wf.handle_turn(
    "1020 Harbor Avenue",
    state=first.state,
    message_history=first.next_message_history,
)

expected_path = wf.DOCS_ROOT_DIR / "sops" / "east_branch_1.docx"

for result in (one_turn, two_turn):
    assert result.event_type == "final_answer", result.assistant_text
    assert result.state.locked_on_sop
    assert Path(result.state.active_sop_path) == expected_path
    assert "written approval" in result.assistant_text.casefold()
    print(result.assistant_text)

Cross OpCo is allowed for Marigold Kitchens at 1010 Harbor Avenue, East Harbor, but only after the East Branch Trade Compliance lead gives written approval. The approval must be saved in the entry record before transferring the entry.

[Locked to the current SOP. Say 'switch company' to start a new lookup.]
Cross OpCo is allowed for Marigold Kitchens at 1020 Harbor Avenue, but only after the East Branch Trade Compliance lead gives written approval. The approval must be saved in the entry record before transferring the entry.

[Locked to the current SOP. Say 'switch company' to start a new lookup.]


In [1]:
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # app.workflow initializes its OpenAI client on import.

import app.workflow_new as wf

data_root = Path("data").resolve()
wf.ROOT_INDEX_PATH = str(data_root / "index.xlsx")
wf.DOCS_ROOT_DIR = data_root

# Temporary fixture adapter: the current library resolver strips "sops/".
# Keep that subdirectory when resolving links from the generated workbooks.
def resolve_fixture_link(raw_path: str) -> str:
    path = (data_root / str(raw_path).strip()).resolve()
    path.relative_to(data_root)  # Reject paths outside the fixture directory.
    return str(path)

wf.normalize_doc_path = resolve_fixture_link

# result = await wf.handle_turn(
#     user_input="Is Cross OpCo allowed for Alder Supply?",
#     state=None,
#     message_history=[],
# )

# print("Event:", result.event_type)
# print("Selected SOP:", result.state.active_sop_path)
# print("Locked:", result.state.locked_on_sop)
# print("Answer:", result.assistant_text)

In [2]:
result = await wf.handle_turn(
    user_input=(
        "Is Cross OpCo allowed for Marigold Kitchens 1010 Harbor Avenue, East Harbor?"
    ),
    state=None,  # Fresh session; don't reuse the Alder Supply lock.
    message_history=[],
)

print("Event:", result.event_type)
print("Selected SOP:", result.state.active_sop_path)
print("Locked:", result.state.locked_on_sop)
print("Answer:", result.assistant_text)

Event: ask_user
Selected SOP: None
Locked: False
Answer: There are multiple rows for Marigold Kitchens with different documents. Do you want information from the first or second East Harbor branch document?


In [6]:
result = await wf.handle_turn(
    user_input="What should North Harbor Company 01 do if the invoice is missing before arrival at 1210 Cedar Avenue, North Harbor?",
    state=None,  # Fresh session: don't reuse the Marigold Kitchens lock.
    message_history=[],
)

# assert result.event_type == "final_answer", result.assistant_text
# assert not result.state.in_secondary_index
# assert not result.state.awaiting_user_clarification
# assert result.state.locked_on_sop
# # assert result.state.locked_company_name == "Alder Supply"

# # expected_sop = (wf.DOCS_ROOT_DIR / "sops" / "direct_001.docx").resolve()
# # assert result.state.active_sop_path is not None
# # assert Path(result.state.active_sop_path).resolve() == expected_sop

# answer = result.assistant_text.casefold()
# assert "not allowed" in answer or "not permitted" in answer, result.assistant_text

print(result.assistant_text)

If the invoice is missing before arrival at 1210 Cedar Avenue, North Harbor (North Harbor Company 01), they should confirm the importer ID and release status, then place the missing invoice record in the North Branch 1 hold queue.

[Locked to the current SOP. Say 'switch company' to start a new lookup.]


In [3]:
result = await wf.handle_turn(
    user_input="1010 Harbor Avenue, East Harbor",
    state=result.state,
    message_history=result.next_message_history,
)

print("Event:", result.event_type)
print("Selected SOP:", result.state.active_sop_path)
print("Locked:", result.state.locked_on_sop)
print("Answer:", result.assistant_text)

AttributeError: 'ChatTurnResult' object has no attribute 'next_message_history'

In [7]:
from pathlib import Path
import pandas as pd

index_path = Path(result.state.current_index_path)
print("Current index:", index_path)
print("In secondary index:", result.state.in_secondary_index)

df = pd.read_excel(index_path)
df.columns = df.columns.str.strip()

company_rows = df.loc[
    df["company name"].astype(str).str.strip().str.casefold()
    .eq("marigold kitchens")
]
display(company_rows)

if "address" in company_rows.columns:
    matches = company_rows.loc[
        company_rows["address"].astype(str).str.strip().str.casefold()
        .eq("1010 harbor avenue, east harbor")
    ]
    for link in matches["link"]:
        sop_path = Path(wf.normalize_doc_path(link))
        print("Workbook link:", link)
        print("Resolved path:", sop_path)
        print("DOCX exists:", sop_path.is_file())

Current index: /home/aytu/apps/docsearch/data/second_index_east.xlsx
In secondary index: True


,company name,address,link
0,Marigold Kitchens,"1010 Harbor Avenue, East Harbor",sops/east_branch_1.docx
1,Marigold Kitchens,"1020 Harbor Avenue, East Harbor",sops/east_branch_1.docx
2,Marigold Kitchens,"1030 Harbor Avenue, East Harbor",sops/east_branch_2.docx


Workbook link: sops/east_branch_1.docx
Resolved path: /home/aytu/apps/docsearch/data/sops/east_branch_1.docx
DOCX exists: True


In [3]:
from __future__ import annotations

from pathlib import Path
from typing import Literal, Any
import re
import pandas as pd
from docx import Document
from pydantic import BaseModel, ConfigDict
from pydantic_ai import Agent


# -----------------------------
# CONFIG
# -----------------------------
ROOT_INDEX_WINDOWS = r"C:\Users\maytu\OneDrive\Desktop\docs\index.xlsx"


# -----------------------------
# HELPERS
# -----------------------------
def normalize_windows_path(raw_path: str) -> str:
    raw_path = str(raw_path).strip()
    if re.match(r"^[A-Za-z]:\\", raw_path):
        drive = raw_path[0].lower()
        rest = raw_path[2:].replace("\\", "/")
        return f"/mnt/{drive}{rest}"
    return raw_path


ROOT_INDEX_PATH = normalize_windows_path(ROOT_INDEX_WINDOWS)


def load_small_table(index_path: str) -> pd.DataFrame:
    path = Path(index_path)
    if not path.exists():
        raise FileNotFoundError(f"Index file not found: {path}")

    suffix = path.suffix.lower()
    if suffix == ".xlsx":
        df = pd.read_excel(path, engine="openpyxl")
    elif suffix == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported table type: {suffix}")

    df = df.fillna("")
    df.columns = [str(c).strip() for c in df.columns]
    return df.reset_index(drop=True)


def dataframe_to_row_text(df: pd.DataFrame) -> str:
    lines = []
    for i, row in df.iterrows():
        cells = [f"{col}={str(row[col]).strip()}" for col in df.columns]
        lines.append(f"row_id={i} | " + " | ".join(cells))
    return "\n".join(lines)


def extract_link_from_row(row: pd.Series) -> str | None:
    for value in row.values:
        if isinstance(value, str):
            v = value.strip()
            if v.lower().endswith(".docx") or v.lower().endswith(".xlsx") or v.lower().endswith(".csv"):
                return v
    return None


def read_docx_text(path: str) -> str:
    doc = Document(path)
    return "\n".join(
        p.text.strip()
        for p in doc.paragraphs
        if p.text and p.text.strip()
    )


def wants_new_search(text: str) -> bool:
    t = text.lower().strip()
    triggers = [
        "switch company",
        "new company",
        "different company",
        "change company",
        "search again",
        "look up another",
        "reset",
        "start over",
    ]
    return any(x in t for x in triggers)


# -----------------------------
# MODELS
# -----------------------------
class WorkflowState(BaseModel):
    conversation_id: str
    original_user_query: str
    current_index_path: str

    awaiting_user_clarification: bool = False
    clarification_question: str | None = None

    selected_link: str | None = None
    active_sop_path: str | None = None
    locked_company_name: str | None = None
    locked_on_sop: bool = False

    status: str = "searching_index"


class TableDecision(BaseModel):
    action: Literal["select_row", "ask_user", "no_match"]
    selected_row_id: int | None = None
    question: str | None = None


class SopAnswer(BaseModel):
    answer: str


class ChatTurnResult(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    event_type: Literal["ask_user", "final_answer", "error"]
    assistant_text: str
    state: WorkflowState
    raw_result: Any = None


# -----------------------------
# AGENTS
# -----------------------------
match_agent = Agent(
    "openai:gpt-4.1-mini",
    output_type=TableDecision,
    system_prompt=(
        "You are selecting a row from a small company table. "
        "Return select_row if one row clearly matches the user's request. "
        "Return ask_user with exactly one short clarification question if there is ambiguity. "
        "Return no_match if nothing fits."
    ),
)

qa_agent = Agent(
    "openai:gpt-4.1-mini",
    output_type=SopAnswer,
    system_prompt=(
        "You answer questions only from the SOP text provided. "
        "Be concise. If the answer is not in the SOP, say that clearly. "
        "Do not switch companies or speculate. Follow these rules strictly:"
        """
        Rules:
- The SOP has already been selected by the application.
- Answer only from the SOP text below.
- Do not say the SOP is missing for another company name mentioned earlier.
- Do not re-evaluate which company was intended.
- If the answer is in the SOP, give it directly.
- If the answer is not in the SOP, say: "That information is not in the selected SOP."

        """
    ),
)


# -----------------------------
# SOP QA
# -----------------------------
async def answer_from_sop(user_question: str, sop_path: str) -> SopAnswer:
    sop_text = read_docx_text(sop_path)

    prompt = f"""
User question:
{user_question}

SOP text:
{sop_text}
"""
    result = await qa_agent.run(prompt)
    return result.output


# -----------------------------
# MAIN WORKFLOW
# -----------------------------
async def handle_turn(
    user_input: str,
    state: WorkflowState | None,
    message_history=None,
) -> ChatTurnResult:
    if message_history is None:
        message_history = []

    if state is None:
        state = WorkflowState(
            conversation_id="demo-1",
            original_user_query=user_input,
            current_index_path=ROOT_INDEX_PATH,
            status="searching_index",
        )

    elif state.locked_on_sop and state.active_sop_path and not wants_new_search(user_input):
        sop_result = await answer_from_sop(user_input, state.active_sop_path)
        return ChatTurnResult(
            event_type="final_answer",
            assistant_text=sop_result.answer,
            state=state,
            raw_result=None,
        )

    elif state.locked_on_sop and wants_new_search(user_input):
        state.locked_on_sop = False
        state.active_sop_path = None
        state.locked_company_name = None
        state.selected_link = None
        state.awaiting_user_clarification = False
        state.clarification_question = None
        state.current_index_path = ROOT_INDEX_PATH
        state.status = "searching_index"
        state.original_user_query = user_input

    elif state.awaiting_user_clarification:
        state.original_user_query += f"\nUser clarification: {user_input}"
        state.awaiting_user_clarification = False
        state.clarification_question = None

    else:
        state.original_user_query = user_input

    while True:
        df = load_small_table(state.current_index_path)
        table_text = dataframe_to_row_text(df)

        prompt = f"""
User query:
{state.original_user_query}

Current table:
{table_text}

Your task:
- Choose the best matching row if clear.
- If ambiguous, ask one short clarification question.
- If no row matches, return no_match.
"""

        agent_result = await match_agent.run(
            prompt,
            message_history=message_history,
        )
        decision = agent_result.output

        if decision.action == "ask_user":
            state.awaiting_user_clarification = True
            state.clarification_question = decision.question
            state.status = "needs_clarification"

            return ChatTurnResult(
                event_type="ask_user",
                assistant_text=decision.question,
                state=state,
                raw_result=agent_result,
            )

        if decision.action == "no_match":
            state.status = "error"
            return ChatTurnResult(
                event_type="error",
                assistant_text="I could not identify the right row in the table.",
                state=state,
                raw_result=agent_result,
            )

        if decision.selected_row_id is None or decision.selected_row_id >= len(df):
            state.status = "error"
            return ChatTurnResult(
                event_type="error",
                assistant_text="The model selected an invalid row.",
                state=state,
                raw_result=agent_result,
            )

        selected_row = df.iloc[decision.selected_row_id]
        link = extract_link_from_row(selected_row)

        if not link:
            state.status = "error"
            return ChatTurnResult(
                event_type="error",
                assistant_text="The selected row did not contain a usable file link.",
                state=state,
                raw_result=agent_result,
            )

        resolved_link = normalize_windows_path(link)

        if resolved_link.lower().endswith((".xlsx", ".csv")):
            state.current_index_path = resolved_link
            state.status = "searching_index"
            message_history = agent_result.all_messages()
            continue

        if resolved_link.lower().endswith(".docx"):
            state.selected_link = resolved_link
            state.active_sop_path = resolved_link
            state.locked_company_name = str(selected_row.to_dict())
            state.locked_on_sop = True
            state.status = "locked_on_sop"

            sop_result = await answer_from_sop(
                state.original_user_query,
                state.active_sop_path,
            )

            locked_note = "\n\n[Locked to current SOP. Say 'switch company' to search another company.]"

            return ChatTurnResult(
                event_type="final_answer",
                assistant_text=sop_result.answer + locked_note,
                state=state,
                raw_result=agent_result,
            )

        state.status = "error"
        return ChatTurnResult(
            event_type="error",
            assistant_text="Selected row did not contain a usable DOCX, XLSX, or CSV link.",
            state=state,
            raw_result=agent_result,
        )

/home/aytu/apps/docsearch/.venv/lib/python3.11/site-packages/pydantic_ai/agent/__init__.py:410: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


In [4]:
import gradio as gr

def init_session():
    return {
        "workflow_state": None,
        "message_history": [],
    }

async def respond(message, chat_history, session):
    if not message or not message.strip():
        return chat_history, session, {}, ""

    if session is None:
        session = init_session()

    result = await handle_turn(
        user_input=message,
        state=session["workflow_state"],
        message_history=session["message_history"],
    )

    session["workflow_state"] = result.state

    if result.raw_result is not None:
        try:
            session["message_history"] = result.raw_result.all_messages()
        except Exception:
            session["message_history"] = []

    chat_history = chat_history or []
    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": result.assistant_text},
    ]

    debug_payload = {
        "event_type": result.event_type,
        "assistant_text": result.assistant_text,
        "workflow_state": result.state.model_dump(),
        "message_count": len(session["message_history"]),
    }

    return chat_history, session, debug_payload, ""

def reset_session():
    return [], init_session(), {}, ""

with gr.Blocks() as demo:
    gr.Markdown("## SOP Bot")

    chatbot = gr.Chatbot(height=500)
    msg = gr.Textbox(
        label="Ask about a company or SOP",
        placeholder="Example: I need the SOP for ccc",
    )
    send = gr.Button("Send")
    clear = gr.Button("Reset")
    session = gr.State(init_session())
    debug = gr.JSON(label="Debug State")

    send.click(
        respond,
        inputs=[msg, chatbot, session],
        outputs=[chatbot, session, debug, msg],
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot, session],
        outputs=[chatbot, session, debug, msg],
    )

    clear.click(
        reset_session,
        inputs=[],
        outputs=[chatbot, session, debug, msg],
    )

demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


In [1]:
from app.workflow import handle_turn, WorkflowState

import app.workflow as wf
from pathlib import Path

wf.ROOT_INDEX_PATH = "/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx"
wf.CONTAINER_DOCS_DIR = Path("/mnt/c/Users/maytu/OneDrive/Desktop/docs")
# wf.DOCS_ROOT_DIR = Path("/data/docs")
wf.DOCS_ROOT_DIR=Path("/mnt/c/Users/maytu/OneDrive/Desktop/docs")


from pathlib import Path
Path("/mnt/c/Users/maytu/OneDrive/Desktop/docs/second_index.xlsx").exists()

True

In [24]:
result = await handle_turn(
    user_input="hello. how are you",
    state=None,
    message_history=[],
)

result

ChatTurnResult(event_type='final_answer', assistant_text='Hi — I can help you find the right SOP. Tell me the company name or alternate name.', state=WorkflowState(conversation_id='demo-1', original_user_query='hello. how are you', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', awaiting_user_clarification=False, clarification_question=None, selected_link=None, active_sop_path=None, locked_company_name=None, locked_on_sop=False, in_secondary_index=False, has_asked_secondary_clarification=False, pending_user_question=None, parsed_mentions=[], status='searching_index'), next_message_history=[ModelRequest(parts=[SystemPromptPart(content="You parse the user's turn into a structured object for SOP search. Do not answer the SOP question. A company name, alternate name, or identifier counts as a usable lookup reference. If the user already provided a concrete company name, alternate name, or identifier, set retrieval_status='ready'. Do not ask for the company again i

In [13]:
result = await handle_turn(
    user_input="what is the color code for alternate company name 21_5, if temp is above 55F?",#"can you help me find the SOP for ccc company?",
    state=result.state,
    message_history=result.next_message_history,
)

result

ChatTurnResult(event_type='error', assistant_text='I could not confidently identify the right row.', state=WorkflowState(conversation_id='demo-1', original_user_query='what is the sop for ccc\nClarification: company 1 variant 3\nClarification: what is the color code for alternate company name 21_5, if temp is above 55F?', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/second_index.xlsx', awaiting_user_clarification=False, clarification_question=None, selected_link=None, active_sop_path=None, locked_company_name=None, locked_on_sop=False, in_secondary_index=True, has_asked_secondary_clarification=True, pending_user_question='what is the sop for ccc', parsed_mentions=[EntityMention(surface_text='ccc', normalized_text=None, role='company_name', can_lookup_sop=True)], status='error'), next_message_history=[ModelRequest(parts=[SystemPromptPart(content="You decide which table row best matches a row-lookup request after hybrid retrieval has already produced candidates. You will 

In [25]:
result = await handle_turn(
    user_input="what is the sop for ccc",#"can you help me find the SOP for ccc company?",
    state=result.state,
    message_history=result.next_message_history,
)

result

ChatTurnResult(event_type='ask_user', assistant_text="There are three entries for 'ccc' with different notes. Which note or variant of 'ccc' are you interested in?", state=WorkflowState(conversation_id='demo-1', original_user_query='what is the sop for ccc', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', awaiting_user_clarification=True, clarification_question="There are three entries for 'ccc' with different notes. Which note or variant of 'ccc' are you interested in?", selected_link=None, active_sop_path=None, locked_company_name=None, locked_on_sop=False, in_secondary_index=False, has_asked_secondary_clarification=False, pending_user_question='what is the sop for ccc', parsed_mentions=[EntityMention(surface_text='ccc', normalized_text=None, role='company_name', can_lookup_sop=True)], status='needs_clarification'), next_message_history=[ModelRequest(parts=[SystemPromptPart(content="You decide which table row best matches a row-lookup request after hybrid re

In [26]:
result = await handle_turn(
    user_input="company 1 variant 3",#"can you help me find the SOP for ccc company?",
    state=result.state,
    message_history=result.next_message_history,#result.raw_result.all_messages() if hasattr(result.raw_result, "all_messages") else [],
)

result

ChatTurnResult(event_type='ask_user', assistant_text="Do you want the company 'ccc' located at '1 osmanli caddesi' or at '1 selcuklu caddesi'?", state=WorkflowState(conversation_id='demo-1', original_user_query='what is the sop for ccc\nClarification: company 1 variant 3', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/second_index.xlsx', awaiting_user_clarification=True, clarification_question="Do you want the company 'ccc' located at '1 osmanli caddesi' or at '1 selcuklu caddesi'?", selected_link=None, active_sop_path=None, locked_company_name=None, locked_on_sop=False, in_secondary_index=True, has_asked_secondary_clarification=True, pending_user_question='what is the sop for ccc', parsed_mentions=[EntityMention(surface_text='ccc', normalized_text=None, role='company_name', can_lookup_sop=True)], status='needs_clarification'), next_message_history=[ModelRequest(parts=[SystemPromptPart(content="You decide which table row best matches a row-lookup request after hybrid ret

In [27]:
result = await handle_turn(
    user_input="1 selcuklu cad",#"can you help me find the SOP for ccc company?",
    state=result.state,
    message_history=result.next_message_history#result.raw_result.all_messages() if hasattr(result.raw_result, "all_messages") else [],
)

result

ChatTurnResult(event_type='final_answer', assistant_text="The SOP for CCC in the Selcuklu company is to use key 1071.\n\n[Locked to current SOP. Say 'switch company' to search another company.]", state=WorkflowState(conversation_id='demo-1', original_user_query='what is the sop for ccc\nClarification: company 1 variant 3\nClarification: 1 selcuklu cad', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/second_index.xlsx', awaiting_user_clarification=False, clarification_question=None, selected_link='/mnt/c/Users/maytu/OneDrive/Desktop/docs/4.docx', active_sop_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/4.docx', locked_company_name='ccc', locked_on_sop=True, in_secondary_index=True, has_asked_secondary_clarification=True, pending_user_question='what is the sop for ccc', parsed_mentions=[EntityMention(surface_text='ccc', normalized_text=None, role='company_name', can_lookup_sop=True)], status='locked_on_sop'), next_message_history=[ModelRequest(parts=[SystemPromptPart(conte

In [7]:
result = await handle_turn(
    user_input="temp is 55F",#"can you help me find the SOP for ccc company?",
    state=result.state,
    message_history=result.raw_result.all_messages() if result.raw_result is not None else [],
)

result

ChatTurnResult(event_type='final_answer', assistant_text='If the temperature is 55F, according to the SOP, the color is red.', state=WorkflowState(conversation_id='demo-1', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', awaiting_user_clarification=False, clarification_question=None, selected_link='/mnt/c/Users/maytu/OneDrive/Desktop/docs/company_21_5.docx', active_sop_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/company_21_5.docx', locked_company_name='company_21', locked_on_sop=True, status='locked_on_sop', last_user_query='temp is 55F', last_sop_search_query='sop for twenty one\nUser clarification: sop for twenty one dash 5', root_company_name=None, in_secondary_index=False, secondary_index_path=None), raw_result=None)

In [6]:
result = await handle_turn(
    user_input="variant 2",
    state=result.state,
    message_history=result.raw_result.all_messages() if result.raw_result is not None else [],
)

result

ChatTurnResult(event_type='ask_user', assistant_text="There are two SOP documents for company 'ccc'. Are you looking for the one located at '/mnt/c/Users/maytu/OneDrive/Desktop/docs/3.docx' or '/mnt/c/Users/maytu/OneDrive/Desktop/docs/4.docx'?", state=WorkflowState(conversation_id='demo-1', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/second_index.xlsx', awaiting_user_clarification=True, clarification_question="There are two SOP documents for company 'ccc'. Are you looking for the one located at '/mnt/c/Users/maytu/OneDrive/Desktop/docs/3.docx' or '/mnt/c/Users/maytu/OneDrive/Desktop/docs/4.docx'?", selected_link=None, active_sop_path=None, locked_company_name=None, locked_on_sop=False, status='needs_clarification', last_user_query='can you help me find the SOP for ccc company?\nUser clarification: variant 2', last_sop_search_query='can you help me find the SOP for ccc company?\nUser clarification: variant 2', root_company_name='ccc', in_secondary_index=True, secondary_

In [7]:
result = await handle_turn(
    user_input="1 selcuklu caddesi is the address",
    state=result.state,
    message_history=result.raw_result.all_messages() if result.raw_result is not None else [],
)

result

ChatTurnResult(event_type='final_answer', assistant_text='I found and opened the SOP for ccc. This SOP is now selected for the session.\n\nYou can now ask me questions about this SOP, ask general chat questions, or ask for a different company.', state=WorkflowState(conversation_id='demo-1', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/second_index.xlsx', awaiting_user_clarification=False, clarification_question=None, selected_link='/mnt/c/Users/maytu/OneDrive/Desktop/docs/4.docx', active_sop_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/4.docx', locked_company_name='ccc', locked_on_sop=True, status='locked_on_sop', last_user_query='can you help me find the SOP for ccc company?\nUser clarification: variant 2\nUser clarification: 1 selcuklu caddesi is the address', last_sop_search_query='can you help me find the SOP for ccc company?\nUser clarification: variant 2\nUser clarification: 1 selcuklu caddesi is the address', root_company_name='ccc', in_secondary_index=False, s

In [8]:
result = await handle_turn(
    user_input="what key should i use",
    state=result.state,
    message_history=result.raw_result.all_messages() if result.raw_result is not None else [],
)

result

ChatTurnResult(event_type='final_answer', assistant_text='You should use key 1071 according to the SOP of Selcuklu company.', state=WorkflowState(conversation_id='demo-1', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/second_index.xlsx', awaiting_user_clarification=False, clarification_question=None, selected_link='/mnt/c/Users/maytu/OneDrive/Desktop/docs/4.docx', active_sop_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/4.docx', locked_company_name='ccc', locked_on_sop=True, status='locked_on_sop', last_user_query='what key should i use', last_sop_search_query='can you help me find the SOP for ccc company?\nUser clarification: variant 2\nUser clarification: 1 selcuklu caddesi is the address', root_company_name='ccc', in_secondary_index=False, secondary_index_path=None), raw_result=None)

In [6]:
from pathlib import Path

p = Path("/data/docs/company_21_5.docx")
print("exists:", p.exists())
print("is_file:", p.is_file())
print("suffix:", p.suffix)

exists: False
is_file: False
suffix: .docx


In [8]:
r1 = result

r2 = await handle_turn(
    user_input="tewnty one dash five",
    state=r1.state,
    message_history=r1.raw_result.all_messages() if r1.raw_result is not None else [],
)

r2

ChatTurnResult(event_type='final_answer', assistant_text='I found and opened the SOP for company_21. This SOP is now selected for the session.\n\nYou can now ask me questions about this SOP, ask general chat questions, or ask for a different company.', state=WorkflowState(conversation_id='demo-1', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', awaiting_user_clarification=False, clarification_question=None, selected_link='/mnt/c/Users/maytu/OneDrive/Desktop/docs/company_21_5.docx', active_sop_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/company_21_5.docx', locked_company_name='company_21', locked_on_sop=True, status='locked_on_sop', last_user_query='sop for twenty one\nUser clarification: sop for twenty one dash 5\nUser clarification: tewnty one dash five', last_sop_search_query='sop for twenty one\nUser clarification: sop for twenty one dash 5\nUser clarification: tewnty one dash five', root_company_name=None, in_secondary_index=False, secondary_index_path=

In [15]:
import pandas as pd

companies = [f"company_{i+1}" for i in range(200)]
rows = []

for company in companies:
    for alt_idx in range(1, 6):  # 5 alternate names per company
        alt_name = f"{company}_{alt_idx}"
        notes = f"Test notes for {company}, variant {alt_idx}"
        link = f"/data/docs/{company}_{alt_idx}.docx"
        rows.append({
            "company name": company,
            "alternate name": alt_name,
            "notes": notes,
            "link": link,
        })

df = pd.DataFrame(rows)
print(len(df))  # should be 1000

df.to_excel("mock_index.xlsx", index=False)

1000
